# نظام RAG بالعربي - قانون التأمين الاجتماعي المصري

النسخة العربية المعدلة: تفهم PDF عربي وتجاوب بالعربي

**أهم التعديلات عن النسخة الإنجليزية:**
1. موديل التضمين (Embeddings) بقى متعدد اللغات `intfloat/multilingual-e5-large` بدل الموديل الإنجليزي الافتراضي — دي كانت أكبر مشكلة عندك
2. تنظيف وتوحيد النص العربي (التشكيل، التطويل، أ/إ/آ → ا)
3. تقسيم النص بفواصل عربية (،، ؟) و chunk أكبر مناسب للنصوص القانونية
4. الـ Prompt بالعربي ويجبر الموديل يجاوب بالعربية


In [1]:
# ✅ الخطوة 1: تثبيت المكتبات
!pip install -q langchain langchain-community langchain-core langchain-openai langchain-text-splitters chromadb fastembed pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.

In [2]:
# ✅ الخطوة 2: الاستيرادات
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import os
import re

/tmp/ipykernel_2079/1849835401.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
# ✅ الخطوة 3: رفع الـ PDF
# لو شغال على Colab استخدم السطرين دول، لو شغال محلي حط الملف جنب النوت بوك
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"تم رفع: {pdf_path}")
except ImportError:
    # تشغيل محلي
    pdf_path = "data/law-79-1975.pdf"
    print(f"استخدام الملف المحلي: {pdf_path}")

Saving law-79-1975.pdf to law-79-1975.pdf
تم رفع: law-79-1975.pdf


In [4]:
# ✅ الخطوة 4: دالة تنظيف النص العربي — مهمة جداً للبحث
# المشكلة: النص العربي فيه أشكال متعددة لنفس الحرف (أ، إ، آ، ا) وتشكيل وتطويل (ـ)
# الحل: نوحدهم قبل التخزين عشان البحث يلاقي النتايج صح

def normalize_arabic(text: str) -> str:
    """توحيد النص العربي لتحسين البحث والتطابق."""
    # إزالة التشكيل (فتحة، ضمة، كسرة، شدة...)
    text = re.sub(r'[ً-ٰٟ]', '', text)
    # ازالة التطويل (ـ)
    text = text.replace('ـ', '')
    # توحيد الألف
    text = re.sub(r'[أإآا]', 'ا', text)
    # توحيد الياء
    text = text.replace('ى', 'ي')
    # توحيد التاء المربوطة (اختياري — احذف السطر ده لو عايز دقة قانونية أعلى)
    # text = text.replace('ة', 'ه')
    # تنظيف المسافات والسطور الزيادة
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# نطبقها أيضاً على السؤال قبل البحث
def normalize_query(query: str) -> str:
    return normalize_arabic(query)

In [5]:
# ✅ الخطوة 5: تحميل الـ PDF وتقسيمه
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"عدد الصفحات: {len(docs)}")
print("نموذج من أول صفحة:")
print(docs[0].page_content[:500])

# تنظيف النص العربي قبل التقسيم
for doc in docs:
    doc.page_content = normalize_arabic(doc.page_content)

# ⚠️ مهم: chunk_size أكبر من النسخة الإنجليزية
# النصوص القانونية العربية جملها طويلة، لو قسمت على 500 حرف المادة القانونية هتتقطع
# وضفنا فواصل عربية (، ؟) عشان التقسيم يحترم الجمل العربية
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "،", "؟", ".", ":", ";", " ", ""]
)
splits = splitter.split_documents(docs)
print(f"عدد المقاطع بعد التقسيم: {len(splits)}")
print("نموذج مقطع:")
print(splits[0].page_content[:500])

عدد الصفحات: 64
نموذج من أول صفحة:
قانون رقم97 لسنة5791  
بإصدار قانون التأمين الاجتماعي 
 
باسم الشعب 
رئيس الجمهورية 
قرر مجلس الشعب القانون الأتى نصه ، وقد أصدرناة؛ 
 
(المادة الأولى) 
يعمل فيما يتعلق بنظام التأمين الاجتماعى بأحكام القانون المرافق  . 
(المادة الثانية) 
 
يحل هذا القانون محل التشريعات الآتية : 
     1) الأمر الصادر فى62 من ديسمبر سنة 4581 بشأن المعاشات المدنية . 
     2) الأمر الصادر فى44 من يناير سنة 1871 بشأن المعاشات المدنية . 
     3) الأمر الصادر فى64 من يونيه سنة4551 بشأن المعاشات المدنية . 
     4) القان
عدد المقاطع بعد التقسيم: 215
نموذج مقطع:
قانون رقم97 لسنة5791 باصدار قانون التامين الاجتماعي باسم الشعب رئيس الجمهورية قرر مجلس الشعب القانون الاتي نصه


In [6]:
# ✅ الخطوة 6: التضمين (Embeddings) متعدد اللغات — أهم تعديل!
# المشكلة في كودك القديم: FastEmbedEmbeddings() بدون تحديد الموديل
# بيستخدم BAAI/bge-small-en-v1.5 اللي إنجليزي فقط → فاشل مع العربي
# الحل: موديل intfloat/multilingual-e5-large بيفهم عربي كويس

embedding = FastEmbedEmbeddings(model_name="intfloat/multilingual-e5-large")
# persist_directory يخلي قاعدة البيانات محفوظة على القرص وتظهر في ملفات Colab
# (من غيره القاعدة بتبقى في الرام فقط ومش بتظهر في أي مكان)
vectorstore = Chroma.from_documents(
    splits, embedding=embedding, persist_directory="./chroma_db"
)

# k=5 عشان القانون محتاج سياق أكبر (مواد متعددة)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# ✅ إثبات إن الـ VDB اتعملت وتشوف محتواها:
print(f"عدد المقاطع المخزنة في الـ VDB: {vectorstore._collection.count()}")
print(f"مكان الحفظ على القرص: ./chroma_db")

/usr/local/lib/python3.13/dist-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

عدد المقاطع المخزنة في الـ VDB: 215
مكان الحفظ على القرص: ./chroma_db


In [10]:
# ✅ الخطوة 7: تجهيز موديل اللغة (Groq)
# ⚠️ لا تكتب الـ API Key في الكود مباشرة (كودك القديم كان فيه المفتاح مكشوف!)
from getpass import getpass
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass("Write the API key: ")

# ⚠️ مهم: موديلات الـ reasoning مثل qwen/qwen3 بتطبع تفكيرها داخل <think> قبل الإجابة
# وده اللي كان بيظهر عندك. gpt-oss-120b هو البديل الرسمي من Groq بعد إيقاف llama-3.3
# (اتوقف يوم 2026-08-16) وبيرد بالإجابة النهائية مباشرة بدون تفكير ظاهر.
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=GROQ_API_KEY,
    temperature=0.2,
    max_tokens=1024,
)

Write the API key: ··········


In [11]:
# ✅ الخطوة 8: الـ Prompt بالعربي — تاني أهم تعديل!
# المشكلة في كودك: الـ prompt كان إنجليزي فالموديل كان يرد انجليزي

prompt = PromptTemplate.from_template("""
أنت مساعد قانوني متخصص في قانون التأمين الاجتماعي المصري.
استخدم السياق التالي فقط للإجابة على السؤال.
إذا لم تجد الإجابة في السياق، قل بوضوح: "لا أعرف بناءً على المستند المقدم."
أجب باللغة العربية الفصحى وبشكل واضح ومختصر.
اذكر رقم المادة القانونية عند الإمكان.
ممنوع منعاً باتاً عرض خطوات تفكيرك أو تحليلك — اكتب الإجابة النهائية فقط مباشرة.

السياق:
{context}

السؤال: {input}

الإجابة بالعربية:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# دالة تنظيف الإجابة: تشيل أي بقايا <think> لو الموديل رجعها لأي سبب
def clean_answer(text: str) -> str:
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = text.replace('<think>', '').replace('</think>', '')
    return text.strip()

# دالة السؤال الموحدة: تنظف السؤال + تجيب الإجابة نظيفة + تطبع المقاطع المسترجعة للتحقق
def ask(query: str, show_sources: bool = True):
    docs = retriever.invoke(normalize_query(query))
    raw = rag_chain.invoke(normalize_query(query))
    answer = clean_answer(raw if isinstance(raw, str) else str(raw))
    print("الإجابة:", answer)
    if show_sources:
        print("\n--- المقاطع المسترجعة من الـ VDB (للتحقق) ---")
        for i, d in enumerate(docs, 1):
            src = d.metadata.get('source', '')
            page = d.metadata.get('page', '')
            print(f"[{i}] صفحة {page} | {src}")
            print(d.page_content[:300], "...\n")
    return answer

In [12]:
response = llm.invoke(
    "ما هو نطاق تطبيق قانون التأمين الاجتماعي؟"
)

print(response.content)

**نطاق تطبيق قانون التأمين الاجتماعي**  

قانون التأمين الاجتماعي (أو ما يُعرف أحيانًا بقانون الضمان الاجتماعي) يهدف إلى توفير الحماية الاجتماعية للعمال وأسرهم من مخاطر الفقر والمرض والعجز والشيخوخة. يختلف نطاق تطبيقه من دولة إلى أخرى بحسب التشريعات الوطنية، لكن هناك عناصر مشتركة تُحدِّد الفئات المستفيدة والأنشطة المشمولة. إليك نظرة شاملة على نطاق تطبيق هذا القانون، مع الإشارة إلى أبرز النقاط التي تتضمنها معظم القوانين في المنطقة العربية (مثل قانون التأمين الاجتماعي في مصر، قانون الضمان الاجتماعي في السعودية، إلخ).

---

## 1. الفئات المستهدفة (المستفيدون)

| الفئة | من هو؟ | ملاحظات خاصة |
|-------|--------|--------------|
| **العاملون (الموظفون)** | جميع الأشخاص الذين يعملون بعقود عمل (دوام كامل أو جزئي) لدى صاحب عمل خاص أو حكومي. | يشمل العاملين في القطاع الخاص والعام، سواء كانوا بنظام الأجر الشهري أو بالقطعة. |
| **العاملون في القطاع غير الرسمي** | العمال غير المسجلين رسميًا (مثلاً العمال اليوميين، العاملون في الزراعة الموسمية). | في بعض الدول يُسمح بتسجيلهم طواعية أو عبر برامج خاص

In [13]:
# استخدم دالة ask الموحدة (بتنظف السؤال والإجابة وتعرض المصادر)
ask("ما هي شروط استحقاق المعاش؟")

الإجابة: **شروط استحقاق المعاش وفقاً لقانون التأمين الاجتماعي المصري**  

1. **توفر حالة الاستحقاق** – يجب أن يتحقق أحد الأسباب المنصوص عليها (سن التقاعد، العجز الكامل، أو الوفاة) ويُستحق المعاش من أول شهر يحدث فيه السبب (مادة 21).  

2. **مدة الاشتراك** – لا بد أن يكون للمؤمن عليه مدة اشتراك لا تقل عن **ثلاثة أشهر متصلة أو ستة أشهر متقطعة** (مادة 21، البند 7‑1).  

3. **السن أو مدة الخدمة**  
   - للمتقاعد: بلوغ **سن الستين** أو استكمال **مدة خدمة 36 سنة** (أو ما يحدد الحد الأقصى للمعاش) (مادة 24، مادة 26).  
   - إذا زادت مدة الاشتراك عن 36 سنة أو عن الحد المطلوب للحد الأقصى للمعاش، يُستحق تعويض دفعة واحدة يساوي **48 % من الأجر السنوي لكل سنة زائدة** (مادة 26).  

4. **الحد الأدنى للمعاش** – لا يقل عن **20 جنيهًا شهريًا** في حالات بلوغ سن الشيخوخة أو العجز أو الوفاة (مادة 24).  

5. **الورثة المستحقين** (الأرملة/المطلقة، الزوج، الأولاد، الوالدين، الأخوة):  
   - **الأرملة/المطلقة**: يجب توثيق الزواج أو صدقه قضائيًا، وأن يكون عقد الزواج قبل بلوغ المؤمن عليه/المعاش سن الستين (مادة 511)

'**شروط استحقاق المعاش وفقاً لقانون التأمين الاجتماعي المصري**  \n\n1. **توفر حالة الاستحقاق** – يجب أن يتحقق أحد الأسباب المنصوص عليها (سن التقاعد، العجز الكامل، أو الوفاة) ويُستحق المعاش من أول شهر يحدث فيه السبب (مادة\u202f21).  \n\n2. **مدة الاشتراك** – لا بد أن يكون للمؤمن عليه مدة اشتراك لا تقل عن **ثلاثة أشهر متصلة أو ستة أشهر متقطعة** (مادة\u202f21، البند\u202f7‑1).  \n\n3. **السن أو مدة الخدمة**  \n   - للمتقاعد: بلوغ **سن الستين** أو استكمال **مدة خدمة 36 سنة** (أو ما يحدد الحد الأقصى للمعاش) (مادة\u202f24، مادة\u202f26).  \n   - إذا زادت مدة الاشتراك عن 36 سنة أو عن الحد المطلوب للحد الأقصى للمعاش، يُستحق تعويض دفعة واحدة يساوي **48\u202f% من الأجر السنوي لكل سنة زائدة** (مادة\u202f26).  \n\n4. **الحد الأدنى للمعاش** – لا يقل عن **20 جنيهًا شهريًا** في حالات بلوغ سن الشيخوخة أو العجز أو الوفاة (مادة\u202f24).  \n\n5. **الورثة المستحقين** (الأرملة/المطلقة، الزوج، الأولاد، الوالدين، الأخوة):  \n   - **الأرملة/المطلقة**: يجب توثيق الزواج أو صدقه قضائيًا، وأن يكون عقد الزواج قبل

In [14]:
# ✅ الخطوة 9: اسأل بالعربي الآن!
ask("ما هو نطاق تطبيق قانون التأمين الاجتماعي؟")

الإجابة: نطاق تطبيق قانون التأمين الاجتماعي يقتصر على الفئات التالية وفقاً للمادة 2 والمادة 3 من القانون:

1. **العاملون المدنيون** في الجهاز الإداري للدولة، والهيئات العامة، والمؤسسات العامة، والوحدات الاقتصادية التابعة لها أو أي من هذه الجهات.  
2. **العاملون الخاضعون لأحكام قانون العمل** الذين تتوافر فيهم الشروط التالية (المادة 2‑ب):
   - أن يكون عمر المؤمن عليه **45 سنةً أو أكثر**.  
   - أن تكون علاقة العمل **منتظمة** وفقاً للقرارات الصادرة عن وزير التأمينات (يُستثنى من هذا الشرط عمال المقاولات وعمال الشحن والتفريغ).  
   - بالنسبة للأجانب، لا تقل مدة العقد عن سنة ويجب وجود اتفاقية معاملة بالمثل، مع مراعاة الاتفاقيات الدولية التي صدقت عليها مصر.  
3. **المشتغلون بأعمال خدمة المنازل**، باستثناء من يعمل داخل المنازل الخاصة، ويُحدَّدهم بقرار من وزير التأمينات (المادة 2‑ج).  
4. **المستفيدون الذين سبق تأمينهم** وفقاً لقوانين التأمينات الاجتماعية والتأمين والمعاشات المذكورة في المادة 2‑أ (استثناء من أحكام المادة 6).  
5. **العاملون في تأمين إصابات العمل** الذين تقل أعمارهم عن 45 سنة، ب

'نطاق تطبيق قانون التأمين الاجتماعي يقتصر على الفئات التالية وفقاً للمادة\u202f2\u202fوالمادة\u202f3 من القانون:\n\n1. **العاملون المدنيون** في الجهاز الإداري للدولة، والهيئات العامة، والمؤسسات العامة، والوحدات الاقتصادية التابعة لها أو أي من هذه الجهات.  \n2. **العاملون الخاضعون لأحكام قانون العمل** الذين تتوافر فيهم الشروط التالية (المادة\u202f2‑ب):\n   - أن يكون عمر المؤمن عليه **45 سنةً أو أكثر**.  \n   - أن تكون علاقة العمل **منتظمة** وفقاً للقرارات الصادرة عن وزير التأمينات (يُستثنى من هذا الشرط عمال المقاولات وعمال الشحن والتفريغ).  \n   - بالنسبة للأجانب، لا تقل مدة العقد عن سنة ويجب وجود اتفاقية معاملة بالمثل، مع مراعاة الاتفاقيات الدولية التي صدقت عليها مصر.  \n3. **المشتغلون بأعمال خدمة المنازل**، باستثناء من يعمل داخل المنازل الخاصة، ويُحدَّدهم بقرار من وزير التأمينات (المادة\u202f2‑ج).  \n4. **المستفيدون الذين سبق تأمينهم** وفقاً لقوانين التأمينات الاجتماعية والتأمين والمعاشات المذكورة في المادة\u202f2‑أ (استثناء من أحكام المادة\u202f6).  \n5. **العاملون في تأمين إصابات ال

In [ ]:
ask("ما هي شروط استحقاق المعاش؟")

الإجابة: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Role:** Legal assistant specializing in Egyptian Social Insurance Law.
   - **Constraint 1:** Use *only* the provided context to answer.
   - **Constraint 2:** If the answer isn't in the context, state clearly: "لا أعرف بناءً على المستند المقدم."
   - **Constraint 3:** Answer in formal Arabic, clearly and concisely.
   - **Constraint 4:** Mention the legal article number when possible.
   - **Context:** A fragmented text containing excerpts from Egyptian Social Insurance Law (likely Law No. 148 of 2019 or similar, given article numbers like 514, 511, 516, 519, 24, 21, 26, and references to articles 45, 18, 6, 61). It covers conditions for pension eligibility, calculation, beneficiaries (widow, husband, children), minimum/maximum pension, and specific conditions like marriage documentation, age limits, service duration, etc.
   - **Question:** ما هي شروط استحقاق المعاش؟ (What are the conditions for pension 

In [ ]:
# حلقة تفاعلية للأسئلة المتكررة
while True:
    query = input("اسأل سؤال بالعربي (اكتب 'خروج' للانتهاء): ")
    if query.strip() in ["خروج", "exit", "quit"]:
        break
    ask(query, show_sources=False)
    print()
